In [1]:
import nltk, warnings
warnings.filterwarnings('ignore')
for pkg in ['punk_tab', 'stopwords', 'wordnet', 'movie_reviews']:
  nltk.download(pkg, quiet = True)
print('NLP environment ready!')

[nltk_data] Error loading punk_tab: Package 'punk_tab' not found in
[nltk_data]     index


NLP environment ready!


In [2]:
article = (
    'Artificial intelligence is transforming Indian agriculture. Farmers in Punjab now '
    'use drone imagery to detect crop stress days before it is visible to the eye. Machine learning '
    'models trained on soil and weather data recommend precise irrigation schedules, cutting water use by '
    'thirty percent. Startups in Bengaluru are building Hindi and Kannada voice assistants so that '
    'farmers can query these systems without typing. The government has pilot programmes in '
    'six states to subsidise such tools. Experts caution, however, that connectivity gaps in rural '
    'areas still limit adoption. Training and local-language support remain the deciding factors for success.'
)

print('Fixed article loaded:', len(article.split()), 'words')

Fixed article loaded: 97 words


In [3]:
from nltk.chat.util import Chat, reflections

pairs = [
    [r'hi|hello|hey', ['Hello! I am the KRMU helpdesk bot. Ask me about fees, exams or the library.']],
    [r'my name is (.*)', ['Nice to meet you, %1. How can I help you today?']],
    [r'.*fee.*(deadline|last date).*', ['The fee deadline is the 10th of every month.']],
    [r'.*exam.*date.*', ['Sessional exams begin in the first week of the next month.']],
    [r'.*library.*', ['The library is open 9 AM to 8 PM on all working days.']],
    [r'quit', ['Goodbye! Visit the helpdesk for anything I could not answer.']],
    [r'(.*)', ['Could you rephrase that? I can help with the fees, exams and the library.']]
]

bot = Chat(pairs, reflections)

for msg in ['hello', 'my name is Aarav', 'what is the fee deadline?',
            'when is the exam date', 'can I get the pizza here?', 'quit']:
    print('You :', msg)
    print('Bot :', bot.respond(msg))

You : hello
Bot : Hello! I am the KRMU helpdesk bot. Ask me about fees, exams or the library.
You : my name is Aarav
Bot : Nice to meet you, aarav. How can I help you today?
You : what is the fee deadline?
Bot : The fee deadline is the 10th of every month.
You : when is the exam date
Bot : Sessional exams begin in the first week of the next month.
You : can I get the pizza here?
Bot : Could you rephrase that? I can help with the fees, exams and the library.
You : quit
Bot : Goodbye! Visit the helpdesk for anything I could not answer.


In [4]:
try:
    from transformers import pipeline
    mt = pipeline('translation', model='Helsinki-NLP/opus-mt-en-hi')
    for s in ['My name is Priya and I study data science.',
              'Where is the railway station?',
              'It is raining cats and dogs.']:
        print(s, '\n ->', mt(s)[0]['translation_text'], '\n')
except Exception as e:
    mt = None
    print('Model download (~300 MB) requires internet on first run — execute on your machine.  (', type(e).__name__, ')')
    print('Expected: fluent Hindi for sentences 1-2; the idiom is rendered by MEANING (heavy rain),')
    print('not word-for-word — the key observation to record.')

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

Model download (~300 MB) requires internet on first run — execute on your machine.  ( KeyError )
Expected: fluent Hindi for sentences 1-2; the idiom is rendered by MEANING (heavy rain),
not word-for-word — the key observation to record.


In [6]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk import FreqDist
def ext_sum(text, k=3):
    stop = set(stopwords.words('english'))
    words = [w.lower() for w in word_tokenize(text) if w.isalpha() and w.lower() not in stop]
    freq = FreqDist(words)
    sents = sent_tokenize(text)
    score = {s: sum(freq[w.lower()] for w in word_tokenize(s)) / (len(s.split()) + 1) for s in sents}
    top = sorted(score, key=score.get)[-k:]
    return ' '.join(s for s in sents if s in top)   # original order
summ = ext_sum(article)
print('Original:', len(word_tokenize(article)), 'words |', len(sent_tokenize(article)), 'sentences')
print('Summary :', len(word_tokenize(summ)), 'words')
print(summ)
print('Compression ratio:', round(len(word_tokenize(summ)) / len(word_tokenize(article)), 2))

Original: 107 words | 7 sentences
Summary : 43 words
Artificial intelligence is transforming Indian agriculture. Machine learning models trained on soil and weather data recommend precise irrigation schedules, cutting water use by thirty percent. Experts caution, however, that connectivity gaps in rural areas still limit adoption.
Compression ratio: 0.4


In [8]:
try:
    from transformers import pipeline
    summ_ai = pipeline('summarization')
    abs_out = summ_ai(article, max_length=55, min_length=25, do_sample=False)[0]['summary_text']
    print('ABSTRACTIVE:', abs_out)
    print('EXTRACTIVE :', summ)
    for s in sent_tokenize(abs_out):
        print(s in article, '|', s[:60])   # False => newly composed sentence
except Exception as e:
    print('Model download (~1 GB) requires internet on first run — execute on your machine.  (', type(e).__name__, ')')
    print('Expected: a fluent paraphrase; membership checks print False (sentences are NEW);')
    print('fact-check every generated detail against the source — unsupported details = hallucination.')

Model download (~1 GB) requires internet on first run — execute on your machine.  ( KeyError )
Expected: a fluent paraphrase; membership checks print False (sentences are NEW);
fact-check every generated detail against the source — unsupported details = hallucination.


In [9]:
try:
    import speech_recognition as sr
    r = sr.Recognizer()
    with sr.AudioFile('sample.wav') as src:     # record 10-20 s of your voice, export as WAV
        audio = r.record(src)
    try:
        print('Transcript:', r.recognize_google(audio))
    except sr.UnknownValueError:
        print('Speech was not understood.')
    except sr.RequestError as e:
        print('API unavailable:', e)
except FileNotFoundError:
    print('Place sample.wav beside this notebook (record in class), then re-run.')
    print("Expected: a transcript close to what you said - compute the rough word-error rate")
    print('against your reference sentence and note which words failed (accent? noise?).')
except Exception as e:
    print('SpeechRecognition needs internet for recognize_google — run on your machine. (', type(e).__name__, ')')

SpeechRecognition needs internet for recognize_google — run on your machine. ( ModuleNotFoundError )


In [10]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.lm import Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import ngrams
from nltk.corpus import movie_reviews
sm = SmoothingFunction().method1
ref = [['the', 'cat', 'sat', 'on', 'the', 'mat']]
print('BLEU close    :', round(sentence_bleu(ref, ['the','cat','is','on','the','mat'], smoothing_function=sm), 3))
print('BLEU scrambled:', round(sentence_bleu(ref, ['mat','the','on','sat','cat','the'], smoothing_function=sm), 3))
train_sents = [[w.lower() for w in s if w.isalpha()] for s in movie_reviews.sents()[:3000]]
test_sents  = [[w.lower() for w in s if w.isalpha()] for s in movie_reviews.sents()[3000:3050]]
for n in (2, 3):
    tr, vocab = padded_everygram_pipeline(n, train_sents)
    lm = Laplace(n)
    lm.fit(tr, vocab)
    test_ng = [g for s in test_sents for g in ngrams(pad_both_ends(s, n=n), n)]
    print(f'{n}-gram Laplace perplexity:', round(lm.perplexity(test_ng), 1))

BLEU close    : 0.254
BLEU scrambled: 0.064
2-gram Laplace perplexity: 2903.6
3-gram Laplace perplexity: 5323.3


In [11]:
def assistant(command, text):
    if command == 'chat':
        return bot.respond(text)
    if command == 'summarize':
        return ext_sum(text, k=2)
    if command == 'translate':
        return mt(text)[0]['translation_text'] if mt else '[translation module needs the Q2 model loaded]'
    return 'Unknown command. Use: chat | translate | summarize.'

print(assistant('chat', 'what about the library?'))
print(assistant('summarize', article)[:120], '...')
print(assistant('play', 'music'))

The library is open 9 AM to 8 PM on all working days.
Machine learning models trained on soil and weather data recommend precise irrigation schedules, cutting water use by th ...
Unknown command. Use: chat | translate | summarize.
